# Module B05 — Collections

## Exercise 3: Removing duplicates three ways

Duplicate rows are the most common defect in real data. Two systems exported the
same customer. Somebody submitted the form twice. A file was joined to itself.
The list is right in every particular and there are 1,400 names in it where
there should be 1,180.

There are three standard ways to remove duplicates in Python. They are not three
styles of the same thing. They give **different answers**, they cost different
amounts, and one of them refuses to run on data the others handle.

This notebook writes all three, then puts them side by side. On the way it
teaches the two containers you have not met: the **set** and the **tuple**.

| | |
|---|---|
| Time | About 50 minutes |
| You need | This notebook |
| Comes after | Exercise 2, dictionaries |

---

## 1. The data, and what "duplicate" means

Here is a sign-up list with duplicates in it. Two of them.

In [ ]:
signups = ["ama", "kofi", "yaa", "ama", "kwame", "yaa"]

print(signups)
print("rows:", len(signups))

Six rows, four people. Two items are duplicates of items earlier in the list.

Say plainly what you mean by duplicate before you remove any, because Python's
answer is exact equality and yours might not be. `"Ama"` is not equal to `"ama"`.
`"ama "` is not equal to `"ama"`. Every one of the three ways below uses Python's
`==`, so all three will keep `"Ama"` and `"ama"` as two people.

That is not a bug in the technique. It is the technique doing what you asked. If
you meant something looser, cleaning the values comes first and deduplicating
comes second.

---

## 2. Way one: keep the ones you have not seen

The first way uses only what module B04 and exercise 1 gave you: the accumulator
pattern, and `in`.

In [ ]:
signups = ["ama", "kofi", "yaa", "ama", "kwame", "yaa"]

unique = []
for name in signups:
    if name not in unique:
        unique.append(name)

print(unique)
print("people:", len(unique))

Four names, in the order they first appeared. `"ama"` sits where the first `"ama"`
was, not where the second one was, which is usually what a person expects from a
list of sign-ups.

There is a cost hidden in the middle line. `name not in unique` looks through
`unique` one item at a time, and `unique` grows as the loop runs. Over `n` rows
that is roughly `n * n / 2` comparisons, which is the multiplication module B04
exercise 4 warned about. On 1,400 rows it is around a million comparisons and you
will not notice. On 200,000 rows it is twenty billion and you will go home.

Sections 3 to 5 are the container that removes that cost.

---

## 3. A set holds unique items and no positions

A **set** is a collection with two rules: no duplicates, and no order.

```
attendees = {"ama", "kofi", "yaa"}
     │       │  │
     │       │  └── the items, separated by commas
     │       └── curly braces, the same ones a dict uses
     └── an ordinary name
```

Curly braces with `key: value` pairs make a dictionary. Curly braces with plain
items make a set. `set(x)` builds one from anything you can loop through, which
is how you turn a list into a set.

In [ ]:
attendees = {"ama", "kofi", "yaa"}
print(attendees)
print("how many:", len(attendees))

from_a_list = set(["ama", "kofi", "ama", "yaa"])
print(from_a_list)
print("how many:", len(from_a_list))

Four items went in and three came out. A set does not complain about a duplicate,
it does not report one, and it does not keep one. It holds each distinct value
once.

Now the part that catches everybody. **Print a set twice and the items may not
appear in the same arrangement, and that arrangement is not the order you wrote
them in.** A set has no order at all, so there is nothing for it to show you.

- `attendees[0]` raises `TypeError`, because there is no position 0 to ask for.
- Slicing a set is impossible for the same reason.
- Two sets holding the same items are equal, whatever arrangement they print in.

In [ ]:
a = {"ama", "kofi", "yaa"}
b = {"yaa", "ama", "kofi"}

print("equal?", a == b)
print("ama in a?", "ama" in a)

One more piece of syntax, because it bites. `{}` is an **empty dictionary**, not
an empty set. The braces were a dictionary's first. An empty set is written
`set()`.

In [ ]:
print(type({}))
print(type(set()))
print(type({"ama"}))

`.add(item)` puts an item in, and adding one that is already there does nothing
at all rather than raising. `.discard(item)` takes one out and does not mind if
it was never there.

In [ ]:
attendees = set()
attendees.add("ama")
attendees.add("kofi")
attendees.add("ama")          # already in, so nothing happens

print(attendees, "->", len(attendees), "people")

attendees.discard("nobody")   # not there, and that is fine
print(attendees)

---

## 4. Way two: `list(set(items))`

The shortest way to remove duplicates in Python is to make a set and turn it back
into a list.

In [ ]:
signups = ["ama", "kofi", "yaa", "ama", "kwame", "yaa"]

unique = list(set(signups))

print(unique)
print("people:", len(unique))

Four people. Correct count, and look at the arrangement.

That is the wrong answer for a sign-up sheet. Nobody was first. The person who
registered at 9am is somewhere in the middle and there is no way to get the
original order back, because it was thrown away by `set()` and sets do not keep
it.

There is no traceback here and there never will be. The list is the right length
and holds the right names. If your report says "first ten sign-ups", it is now
ten arbitrary sign-ups, and it will keep saying it every day.

Reach for `list(set(x))` when the order genuinely does not matter, and say so out
loud when you write it, because the next person to read the line cannot tell
whether you decided that or forgot to.

---

## 5. Why a set answers `in` without looking

A set is not a list with the duplicates taken out. It stores its items by
computing a number from each value and using that number to decide where it
goes. Finding an item means computing the number again and looking in one place.
There is no walk through the items, and the size of the set makes almost no
difference.

The word for that number is a **hash**, and section 7 is where the word starts
to matter. First, a measurement.

`import time` brings in a piece of Python's standard library. A later module
covers imports properly. One function from it is needed here:
`time.perf_counter()` gives you a number of seconds, and the difference between
two of them is how long something took.

In [ ]:
import time

numbers_list = list(range(20000))
numbers_set = set(numbers_list)

start = time.perf_counter()
for attempt in range(2000):
    19999 in numbers_list
list_seconds = time.perf_counter() - start

start = time.perf_counter()
for attempt in range(2000):
    19999 in numbers_set
set_seconds = time.perf_counter() - start

print("2000 lookups in a list of 20000:", round(list_seconds, 4), "seconds")
print("2000 lookups in a set of 20000: ", round(set_seconds, 6), "seconds")

Your numbers will differ from anybody else's and from your own next run. The gap
will not. The list is doing 2000 walks through 20000 items. The set is doing 2000
single lookups.

Now put that back into section 2. `if name not in unique` over a large list is
the slow line. Change `unique` to a set and it stops being slow, and section 9 is
the version that does that while keeping the order.

---

## 6. Comparing two sets

Sets answer questions about membership between two groups, and they answer them
in one operator each.

In [ ]:
monday = {"ama", "kofi", "yaa"}
tuesday = {"yaa", "kwame", "ama"}

print("came to either day  (union)       ", monday | tuesday)
print("came to both days   (intersection)", monday & tuesday)
print("monday only         (difference)  ", monday - tuesday)
print("tuesday only        (difference)  ", tuesday - monday)

| Operator | Name | Reads as |
|---|---|---|
| `a \| b` | union | in either, or both |
| `a & b` | intersection | in both |
| `a - b` | difference | in `a` and not in `b` |

`a - b` and `b - a` are different questions with different answers, which is
worth noticing before you write one and read the other.

Each operator has a spelled-out name too, and those work on lists directly rather
than needing a set on the right.

In [ ]:
monday = {"ama", "kofi", "yaa"}

print("union:       ", monday.union(["kwame", "ama"]))
print("intersection:", monday.intersection(["yaa", "kofi", "esi"]))
print("difference:  ", monday.difference(["yaa"]))

The alternative to all of that is two nested loops comparing every name against
every name, which is the shape module B04 exercise 4 measured. These operators
are the same job done once, correctly, in a way the next reader can see the
intent of.

---

## 7. Putting a list in a set, run on purpose

Here are the same sign-ups with a second field: the day each person came.

In [ ]:
records = [["ama", "monday"], ["kofi", "monday"], ["ama", "monday"]]

unique = set(records)

```
TypeError: unhashable type: 'list'
```

**`TypeError`** means the type was wrong, and this message names the reason
precisely. A set stores items by their hash, which is a number computed from the
value. **Unhashable** means Python refuses to compute one.

It refuses because a list can change. If `["ama", "monday"]` went into a set, its
hash would decide where it sits. Then somebody appends to it, its value is
different, its hash would be different, and the set is now looking in the wrong
place for an item it is holding. Rather than allow a container that quietly loses
things, Python refuses at the door.

The rule that follows: **only values that cannot change may go in a set, or be
used as a dictionary key.** Text, numbers, and `True`/`False` are all fine. Lists
are not. Dictionaries are not either, for the same reason.

So the fix is not a different set. It is a different kind of group.

---

## 8. A tuple is a group that cannot change

A **tuple** is a fixed group of values. Round brackets instead of square ones.

```
person = ("ama", "monday")
   │      │  │       │
   │      │  └───────┴── the items, in order
   │      └── round brackets
   └── an ordinary name
```

It is indexed and sliced exactly like a list, it has a `len`, and `in` works.
What it does not have is any way to change it: no `.append`, no `.remove`, no
assigning to a position.

In [ ]:
person = ("ama", "monday")

print(person)
print("first:", person[0])
print("last: ", person[-1])
print("len:  ", len(person))
print("is monday in it?", "monday" in person)

Trying to change one is a traceback, and it is worth seeing once.

In [ ]:
person = ("ama", "monday")
person[0] = "kofi"

```
TypeError: 'tuple' object does not support item assignment
```

**Item assignment** is what `person[0] = ...` is called. A list supports it. A
tuple does not, and this is not a limitation that was left out by accident. It is
the entire point of the type.

Because a tuple cannot change, its hash cannot go stale, so a tuple is allowed in
a set and allowed as a dictionary key. That is the fix for section 7.

In [ ]:
records = [("ama", "monday"), ("kofi", "monday"), ("ama", "monday")]

unique = set(records)

print(unique)
print("distinct records:", len(unique))

Three rows, two distinct records, and the duplicate was found by comparing both
fields at once.

Tuples come apart into separate names, which is called **unpacking**. The number
of names on the left has to match the number of items.

In [ ]:
person = ("ama", "monday")

name, day = person
print(name, "came on", day)

for name, day in [("ama", "monday"), ("kofi", "tuesday")]:
    print(name.ljust(6) + day)

That is the same shape you have used twice already without a name for it:
`enumerate` in module B04 hands you a pair each pass, and `.items()` in exercise 2
hands you a pair each pass. Both are handing you tuples.

The brackets are optional when the meaning is clear, so `person = "ama", "monday"`
builds the same tuple. A tuple of one item needs a trailing comma, `("ama",)`,
because otherwise the brackets are only brackets.

**Why immutability is sometimes what you want.** A tuple says, in the syntax, that
this group is a fixed set of fields rather than a collection that grows. A
coordinate is a tuple. A row from a file is a tuple. A shopping list is a list.
And nobody can change a tuple you handed them, which exercise 6 shows is not
something you can say about a list.

---

## 9. Way three: `dict.fromkeys`

Exercise 2 established two facts about dictionaries: **keys are unique**, and
**entries stay in the order you added them**. Together those are exactly what
deduplicating a list needs.

`dict.fromkeys(items)` builds a dictionary using those items as keys. The
duplicates collapse, because keys are unique. The order survives, because
dictionaries keep insertion order. Then `list(...)` gives the keys back as a
list, because looping a dictionary gives its keys.

In [ ]:
signups = ["ama", "kofi", "yaa", "ama", "kwame", "yaa"]

print(dict.fromkeys(signups))

unique = list(dict.fromkeys(signups))
print(unique)
print("people:", len(unique))

Four names, in the order they first appeared, and no loop written by you.

The values are all `None`, which is what `fromkeys` puts there when you do not
give it anything else. You are not using the values, only the keys.

Dictionary keys must be hashable for the same reason set items must, so this way
refuses lists exactly as `set` did. Tuples are fine.

---

## 10. The three ways side by side

Run them together on the same data.

In [ ]:
signups = ["ama", "kofi", "yaa", "ama", "kwame", "yaa"]

one = []
for name in signups:
    if name not in one:
        one.append(name)

two = list(set(signups))

three = list(dict.fromkeys(signups))

print("way one, loop:         ", one)
print("way two, set:          ", two)
print("way three, fromkeys:   ", three)
print()
print("same length?", len(one) == len(two) == len(three))
print("same list?  ", one == two, one == three)

All three found four people. Two of them agree on the answer and one of them
does not, because `==` on lists compares order as well as contents.

| | Keeps original order | Speed on large data | Accepts lists as items |
|---|---|---|---|
| loop with `not in` | yes | slow, `n * n` | yes |
| `list(set(x))` | no | fast | no |
| `list(dict.fromkeys(x))` | yes | fast | no |

`dict.fromkeys` is the one to reach for by default: it is the fast one that is
also the one that keeps the order. The loop earns its place when the items cannot
be hashed, which is the case sets and dictionaries both refuse.

And `list(set(x))` earns its place when the order truly does not matter and you
want the shortest line that says so.

---

## 11. Choosing a container

You now have all four. This is the table to come back to.

| Container | Written | Ordered | Duplicates | Changeable | Looked up by |
|---|---|---|---|---|---|
| list | `[1, 2, 3]` | yes | yes | yes | position |
| tuple | `(1, 2, 3)` | yes | yes | no | position |
| dict | `{"a": 1}` | insertion order | keys unique | yes | key |
| set | `{1, 2, 3}` | no | no | yes | membership only |

Three questions get you to the right row nearly every time.

1. **Do I look things up by a name?** If yes, a dict.
2. **Do I only ever ask "is this in there"?** If yes, a set.
3. **Does the order matter, and will it change?** Changing means a list, fixed
   means a tuple.

Choosing wrongly is not usually a crash. It is a program that works and is
slower than it should be, or one that quietly loses an order somebody needed.
Exercise 5, the written worksheet, is where you practise the choosing.

---

# Your turn

**Do not delete the `# ANSWER n` marker lines.** The self-check uses them.

### Task 1

Fill in every prediction **before you run the cell**.

Do not try to predict the arrangement a set prints in. There is no answer to
that. Predict the counts and the answers to the questions.

In [ ]:
# ANSWER 1
# len(set(rows))            -> ___
# len(list(dict.fromkeys(rows)))  -> ___
# sorted(monday & tuesday)  -> ___
# sorted(monday - tuesday)  -> ___
# sorted(monday | tuesday)  -> ___
# "esi" in (monday | tuesday)  -> ___

rows = ["ama", "kofi", "ama", "yaa", "kofi", "ama"]
monday = {"ama", "kofi", "yaa"}
tuesday = {"yaa", "kwame"}

print("len(set(rows))           ->", len(set(rows)))
print("len(fromkeys)            ->", len(list(dict.fromkeys(rows))))
print("sorted(monday & tuesday) ->", sorted(monday & tuesday))
print("sorted(monday - tuesday) ->", sorted(monday - tuesday))
print("sorted(monday | tuesday) ->", sorted(monday | tuesday))
print('"esi" in either          ->', "esi" in (monday | tuesday))

### Task 2

Write way one: the loop that keeps the ones it has not seen. It must preserve
the order of first appearance.

In [ ]:
# ANSWER 2
signups = ["yaa", "ama", "kofi", "yaa", "esi", "ama", "yaa"]

unique = ___

for name in signups:
    if ___:
        unique.___(name)

print("unique:", unique)
print("kept  :", len(unique), "of", len(signups))

### Task 3

Now ways two and three on the same data, checked against the order the names
first appeared in.

In [ ]:
# ANSWER 3
signups = ["yaa", "ama", "kofi", "yaa", "esi", "ama", "yaa"]
first_appearance = ["yaa", "ama", "kofi", "esi"]

by_set = list(___(signups))
by_fromkeys = list(___(signups))

print("by_set:      ", by_set)
print("by_fromkeys: ", by_fromkeys)

print("by_set found the right people?     ", sorted(by_set) == sorted(___))
print("by_set kept the order?             ", by_set == ___)
print("by_fromkeys kept the order?        ", by_fromkeys == ___)

# Which of the two lost something, and what? ___

### Task 4

Two days of attendance, as lists with duplicates in them. Answer four questions
with set operations rather than loops.

In [ ]:
# ANSWER 4
# This template will not run until you fill the blanks: some of them
# stand where an operator or a keyword goes, not where a value goes.
monday = ["ama", "kofi", "yaa", "ama"]
tuesday = ["yaa", "kwame", "ama", "kwame"]

monday_set = ___(monday)
tuesday_set = ___(tuesday)

print("came on both days: ", sorted(monday_set ___ tuesday_set))
print("came on either day:", sorted(monday_set ___ tuesday_set))
print("monday only:       ", sorted(monday_set ___ tuesday_set))
print("tuesday only:      ", sorted(tuesday_set ___ monday_set))

### Task 5

You have a file of 400,000 order references, exported twice by mistake. The
downstream system processes them in the order they arrive.

Say which of the three ways you would use, why, and what fact about the data
would change your answer.

There is no correct answer here. Naming what the choice depends on is the
exercise.

In [ ]:
# ANSWER 5
which_way_and_why = "___"
what_would_change_my_answer = "___"

# One sentence: what would go wrong if you used list(set(references)) here? ___

---

## Self-check

You do not need to understand this cell. It is machinery, not material.

In [ ]:
def _answer(marker):
    """Find the most recent cell you ran that contains the given marker."""
    try:
        matches = [c for c in _ih if marker in c and "def _answer" not in c]
    except NameError:
        print("Run this in Jupyter or VS Code so the self-check can see your cells.")
        return ""
    return matches[-1] if matches else ""


def check(passed, message):
    print(("PASS  " if passed else "FAIL  ") + message)
    return bool(passed)

NEXT = "Move on to exercise 4, where dictionaries go inside a list and become records."

a1, a2, a3, a4, a5 = (_answer("# ANSWER 1"), _answer("# ANSWER 2"),
                      _answer("# ANSWER 3"), _answer("# ANSWER 4"),
                      _answer("# ANSWER 5"))
tight4 = a4.replace(" ", "")

results = [
    check("len(set(rows))            -> ___" not in a1 and a1.count("___") == 0,
          "Task 1: all six predictions written before running"),
    check(a2.count("___") == 0 and "not in" in a2 and ".append(" in a2,
          "Task 2: the loop keeps only names it has not seen and appends them"),
    check("unique=[]" in a2.replace(" ", ""),
          "Task 2: the accumulator starts as an empty list above the loop"),
    check(a3.count("___") == 0 and "set(" in a3 and "dict.fromkeys(" in a3,
          "Task 3: both of the short ways written"),
    check("lost something, and what? ___" not in a3,
          "Task 3: you named which way lost something and what it lost"),
    check(tight4.count("set(") >= 2 and "&" in tight4 and "|" in tight4
          and tight4.count("-") >= 2,
          "Task 4: union, intersection, and both differences, from sets"),
    check(a4.count("___") == 0, "Task 4: every blank filled"),
    check(a5.count("___") == 0 and "what_would_change_my_answer" in a5,
          "Task 5: a way chosen, a condition that reverses it, and the cost named"),
]

print()
failed = results.count(False)
print("%d of %d checks failing. Keep going." % (failed, len(results)) if failed
      else "All %d checks passing. %s" % (len(results), NEXT))

---

## What you learned

- Deduplicating uses `==`, so `"Ama"` and `"ama"` are two people. Clean first,
  deduplicate second.
- Way one, a loop with `not in`, keeps the original order and gets slow in
  proportion to `n * n`.
- A set holds each value once, has no order and no positions, and `{}` is an
  empty dictionary while `set()` is an empty set.
- Way two, `list(set(x))`, is the shortest and throws the order away with no
  traceback.
- A set finds items by hash rather than by looking through them, which is why
  `in` on a set does not care how large it is.
- `|`, `&`, and `-` give union, intersection, and difference, and `a - b` is a
  different question from `b - a`.
- A list cannot go in a set: `TypeError: unhashable type: 'list'`, because a
  value that can change cannot have a stable hash.
- A tuple is a fixed group in round brackets. It indexes and slices like a list,
  cannot be assigned to, and is allowed in sets and as dictionary keys. Unpacking
  splits one into separate names.
- Way three, `list(dict.fromkeys(x))`, keeps the order and is fast, because keys
  are unique and dictionaries keep insertion order.
- The four containers differ in order, duplicates, changeability, and what you
  look things up by.

## Before you move on

- [ ] You ran `set(records)` on a list of lists and read the word unhashable.
- [ ] You can say what `list(set(x))` loses and why there is no error for it.
- [ ] You can say why a tuple is allowed in a set and a list is not.
- [ ] You have used `&` and `-` instead of writing a nested loop.

**Next:** exercise 4, where dictionaries go inside a list and become the shape
almost all real data arrives in.